In [1]:
EXPERIMENT_NAME = "brats2020_frozen_experiment"
SEED = 42

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 8

PATCH_SIZE = (128, 128, 128)

BATCH_SIZE = 1

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5

Imports & AMP

In [2]:
import os
import json
import random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

Reproducibility Lock

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

Dataset Paths & Patient List

In [4]:
DATASET_PATH = "/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
EXCLUDED_PATIENTS = {"BraTS20_Training_355"}

PATIENTS = sorted([
    pid for pid in os.listdir(DATASET_PATH)
    if pid.startswith("BraTS20_Training_")
    and pid not in EXCLUDED_PATIENTS
])

print("Total patients:", len(PATIENTS))

Total patients: 368


NIfTI Loading

In [5]:
import nibabel as nib

def load_patient(pid):
    pdir = os.path.join(DATASET_PATH, pid)

    def load(fname):
        return nib.load(os.path.join(pdir, fname)).get_fdata()

    flair = load(f"{pid}_flair.nii")
    t1    = load(f"{pid}_t1.nii")
    t1ce  = load(f"{pid}_t1ce.nii")
    t2    = load(f"{pid}_t2.nii")
    seg   = load(f"{pid}_seg.nii")

    image = np.stack([t1, t1ce, t2, flair], axis=0)
    return image, seg

Normalization & Label Mapping

In [6]:
def normalize(image):
    out = np.zeros_like(image)
    for c in range(image.shape[0]):
        x = image[c]
        mask = x > 0
        out[c] = (x - x[mask].mean()) / (x[mask].std() + 1e-8)
    return out

In [7]:
def remap_labels(seg):
    out = np.zeros_like(seg, dtype=np.int64)
    out[seg == 1] = 1
    out[seg == 2] = 2
    out[seg == 4] = 3
    return out

Tumor-Aware Patch Sampler 128³

In [8]:
def sample_patch(image, seg, tumor_prob=0.5):
    _, D, H, W = image.shape
    pd, ph, pw = PATCH_SIZE

    if np.random.rand() < tumor_prob and np.any(seg > 0):
        z, y, x = [v[np.random.randint(len(v))] for v in np.where(seg > 0)]
    else:
        z, y, x = np.random.randint(D), np.random.randint(H), np.random.randint(W)

    z1 = np.clip(z - pd//2, 0, D-pd)
    y1 = np.clip(y - ph//2, 0, H-ph)
    x1 = np.clip(x - pw//2, 0, W-pw)

    return (
        image[:, z1:z1+pd, y1:y1+ph, x1:x1+pw],
        seg[z1:z1+pd, y1:y1+ph, x1:x1+pw]
    )

Dataset Class

In [9]:
class BraTS2020Dataset(Dataset):
    def __init__(self, patient_ids, patches_per_patient):
        self.patient_ids = patient_ids
        self.ppp = patches_per_patient

    def __len__(self):
        return len(self.patient_ids) * self.ppp

    def __getitem__(self, idx):
        pid = self.patient_ids[idx // self.ppp]
        image, seg = load_patient(pid)

        image = normalize(image)
        seg = remap_labels(seg)

        img, lab = sample_patch(image, seg)

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=1).copy()
            lab = np.flip(lab, axis=0).copy()

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=2).copy()
            lab = np.flip(lab, axis=1).copy()

        if np.random.rand() < 0.5:
            img = np.flip(img, axis=3).copy()
            lab = np.flip(lab, axis=2).copy()

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(lab, dtype=torch.long)
        )

Train / Validation Split

In [10]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    PATIENTS, test_size=0.2, random_state=SEED
)

print("Train:", len(train_ids), "Val:", len(val_ids))

Train: 294 Val: 74


DataLoaders

In [11]:
train_loader = DataLoader(
    BraTS2020Dataset(train_ids, patches_per_patient=4),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    BraTS2020Dataset(val_ids, patches_per_patient=2),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

MODEL DEFINITION

In [12]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.size()
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1, 1)
        return x * y

In [14]:
class MBConv3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, expand=4):
        super().__init__()
        mid = in_ch * expand
        self.use_res = in_ch == out_ch and stride == 1

        self.block = nn.Sequential(
            nn.Conv3d(in_ch, mid, 1, bias=False),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),

            nn.Conv3d(mid, mid, 3, stride=stride, padding=1,
                      groups=mid, bias=False),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),

            SEBlock3D(mid),

            nn.Conv3d(mid, out_ch, 1, bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        return x + self.block(x) if self.use_res else self.block(x)

In [15]:
class EfficientNet3DEncoder(nn.Module):
    def __init__(self, in_channels=4, base_c=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, base_c, 3, padding=1),
            nn.BatchNorm3d(base_c),
            nn.ReLU(inplace=True)
        )

        self.b1 = MBConv3D(base_c, base_c)
        self.b2 = MBConv3D(base_c, base_c*2, stride=2)
        self.b3 = MBConv3D(base_c*2, base_c*4, stride=2)
        self.b4 = MBConv3D(base_c*4, base_c*8, stride=2)
        self.b5 = MBConv3D(base_c*8, base_c*16, stride=2)

    def forward(self, x):
        x1 = self.stem(x)
        x2 = self.b1(x1)
        x3 = self.b2(x2)
        x4 = self.b3(x3)
        x5 = self.b4(x4)
        x6 = self.b5(x5)
        return x2, x3, x4, x5, x6

In [16]:
# 5. Up block (no attention)
class Up(nn.Module):
    def __init__(self, in_up, in_skip, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_up, out_ch, 2, stride=2)
        self.conv = DoubleConv(in_skip + out_ch, out_ch)

    def forward(self, x_up, x_skip):
        x_up = self.up(x_up)

        dz = x_skip.size(2) - x_up.size(2)
        dy = x_skip.size(3) - x_up.size(3)
        dx = x_skip.size(4) - x_up.size(4)

        x_up = F.pad(
            x_up,
            [dx//2, dx-dx//2, dy//2, dy-dy//2, dz//2, dz-dz//2]
        )

        return self.conv(torch.cat([x_skip, x_up], dim=1))

In [17]:
class UNet_EfficientNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = EfficientNet3DEncoder()
        self.bot = DoubleConv(512, 1024)

        self.up4 = Up(1024, 256, 512)
        self.up3 = Up(512,  128, 256)
        self.up2 = Up(256,   64, 128)
        self.up1 = Up(128,   32, 64)

        self.outc = nn.Conv3d(64, 4, 1)

    def forward(self, x):
        x2, x3, x4, x5, x6 = self.encoder(x)
        x = self.bot(x6)
        x = self.up4(x, x5)
        x = self.up3(x, x4)
        x = self.up2(x, x3)
        x = self.up1(x, x2)
        return self.outc(x)

In [18]:
model = UNet_EfficientNet3D().cuda()
model = nn.DataParallel(model)

In [19]:
x = torch.randn(1, 4, 128, 128, 128).cuda()
with torch.no_grad():
    y = model(x)

print("Output:", y.shape)

Output: torch.Size([1, 4, 128, 128, 128])


Loss Functions

In [20]:
ce_loss = nn.CrossEntropyLoss()

class DiceLoss(nn.Module):

    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probs = torch.softmax(logits, dim=1)

        targets_onehot = F.one_hot(
            targets,
            num_classes=4
        ).permute(0,4,1,2,3).float()

        dims = (0,2,3,4)

        intersection = torch.sum(
            probs * targets_onehot,
            dims
        )

        cardinality = torch.sum(
            probs + targets_onehot,
            dims
        )

        dice = (
            2 * intersection + self.smooth
        ) / (
            cardinality + self.smooth
        )

        return 1 - dice.mean()

dice_criterion = DiceLoss()
ce_criterion = nn.CrossEntropyLoss()

def combined_loss(logits, targets):

    dice = dice_criterion(
        logits,
        targets
    )

    ce = ce_criterion(
        logits,
        targets
    )

    return dice + ce

In [21]:
def segmentation_metrics(pred, target):

    pred = pred.flatten()
    target = target.flatten()

    tp = np.sum(
        (pred > 0) &
        (target > 0)
    )

    fp = np.sum(
        (pred > 0) &
        (target == 0)
    )

    fn = np.sum(
        (pred == 0) &
        (target > 0)
    )

    dice = (
        2 * tp
    ) / (
        2 * tp + fp + fn + 1e-8
    )

    iou = tp / (
        tp + fp + fn + 1e-8
    )

    precision = tp / (
        tp + fp + 1e-8
    )

    recall = tp / (
        tp + fn + 1e-8
    )

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall
    }

Optimizer & AMP

In [22]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler()

Training & Validation Loops

In [23]:
def train_one_epoch(model, loader):
    model.train()
    running = 0

    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.cuda(), y.cuda()
        optimizer.zero_grad()

        with autocast("cuda"):
            out = model(x)
            loss = combined_loss(out, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()

    return running / len(loader)

In [24]:
def validate(model, loader):

    model.eval()

    metrics = []

    with torch.no_grad():

        for x, y in loader:

            x = x.cuda()
            y = y.cuda()

            logits = model(x)

            pred = torch.argmax(
                logits,
                dim=1
            )

            m = segmentation_metrics(
                pred.cpu().numpy(),
                y.cpu().numpy()
            )

            metrics.append(m)

    return {
        key: np.mean(
            [m[key] for m in metrics]
        )
        for key in metrics[0]
    }

Training Loop + Early Stopping

In [25]:
best_dice = 0.0
patience = 0

epoch_history = []
train_loss_history = []
val_dice_history = []
val_iou_history = []

best_epoch = 0

for epoch in range(MAX_EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    metrics = validate(
        model,
        val_loader
    )

    val_dice = metrics["dice"]
    val_iou = metrics["iou"]
    val_precision = metrics["precision"]
    val_recall = metrics["recall"]

    epoch_history.append(epoch + 1)
    train_loss_history.append(train_loss)
    val_dice_history.append(val_dice)
    val_iou_history.append(val_iou)

    print(
        f"Epoch [{epoch+1}/{MAX_EPOCHS}] "
        f"Loss: {train_loss:.4f} | "
        f"Dice: {val_dice:.4f} | "
        f"IoU: {val_iou:.4f} | "
        f"Precision: {val_precision:.4f} | "
        f"Recall: {val_recall:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice
        best_epoch = epoch + 1
        patience = 0

        torch.save(
            model.module.state_dict(),
            "best_model.pth"
        )

    else:
        patience += 1

    if patience >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

Training:   0%|          | 0/1176 [00:00<?, ?it/s]

Epoch [1/50] Loss: 0.6959 | Dice: 0.7620 | IoU: 0.6753 | Precision: 0.7797 | Recall: 0.7751


Training:   0%|          | 0/1176 [00:00<?, ?it/s]

Epoch [2/50] Loss: 0.4304 | Dice: 0.7912 | IoU: 0.7040 | Precision: 0.8360 | Recall: 0.7898


Training:   0%|          | 0/1176 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788b95d08900>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788b95d08900>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch [10/50] Loss: 0.3417 | Dice: 0.7634 | IoU: 0.6871 | Precision: 0.7643 | Recall: 0.7984


Training:   0%|          | 0/1176 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788b95d08900>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x788b95d08900>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Epoch [11/50] Loss: 0.3316 | Dice: 0.7696 | IoU: 0.6954 | Precision: 0.8205 | Recall: 0.7702
Early stopping triggered.


In [26]:
import pandas as pd

results = pd.DataFrame({
    "epoch": epoch_history,
    "train_loss": train_loss_history,
    "val_dice": val_dice_history,
    "val_iou": val_iou_history
})

results.to_csv(
    "/kaggle/working/training_metrics.csv",
    index=False
)

In [27]:
SAVE_DIR = "/kaggle/working"
MODEL_NAME = "effnet_unet3d_brats2020"   

checkpoint = {
    "model_state_dict": model.module.state_dict(),
    "best_val_dice": best_dice,
    "best_epoch": best_epoch,
    "architecture": "3D U-Net + EfficientNet Encoder",
    "num_classes": 4,
    "input_channels": 4,
    "patch_size": [128, 128, 128],
    "seed": 42
}

torch.save(
    checkpoint,
    os.path.join(SAVE_DIR, f"{MODEL_NAME}.pth")
)

print(f"Model saved as {MODEL_NAME}.pth")

Model saved as effnet_unet3d_brats2020.pth


In [28]:
import json

config = {
    "experiment_name": MODEL_NAME,
    "dataset": {
        "name": "BraTS 2020",
        "modalities": ["T1", "T1ce", "T2", "FLAIR"],
        "excluded_patients": ["BraTS20_Training_355"],
        "label_mapping": {
            "0": "background",
            "1": "necrotic_core",
            "2": "edema",
            "3": "enhancing_tumor"
        }
    },
    "preprocessing": {
        "normalization": "per-modality z-score (non-zero voxels)",
        "patch_size": [128, 128, 128],
        "sampling_strategy": "tumor-aware random patch sampling",
        "tumor_sampling_probability": 0.5
    },
    "data_split": {
        "train_ratio": 0.8,
        "validation_ratio": 0.2,
        "random_seed": 42
    },
    "model": {
        "architecture": checkpoint["architecture"],
        "encoder": "EfficientNet-style MBConv3D" if "effnet" in MODEL_NAME else "Vanilla U-Net",
        "attention": "Yes" if "att" in MODEL_NAME else "No",
        "output_classes": 4
    },
    "training": {
        "optimizer": "Adam",
        "learning_rate": 1e-4,
        "weight_decay": 1e-5,
        "batch_size": 1,
        "loss_function": "Dice + CrossEntropy",
        "mixed_precision": True,
        "max_epochs": 20,
        "early_stopping_patience": 3,
        "best_epoch": best_epoch,
        "best_validation_dice": best_dice
    },
    "hardware": {
        "platform": "Kaggle",
        "gpu": "2 × NVIDIA T4",
        "framework": "PyTorch"
    },
    "reproducibility": {
        "seed": 42,
        "cudnn_deterministic": True
    }
}

config_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved as {MODEL_NAME}_config.json")

Config saved as effnet_unet3d_brats2020_config.json


In [29]:
training_logs = {
    "epochs": epoch_history,
    "train_loss": train_loss_history,
    "val_dice": val_dice_history,
    "val_iou": val_iou_history
}

log_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_training_logs.json")
with open(log_path, "w") as f:
    json.dump(training_logs, f, indent=4)

print(f"Training logs saved as {MODEL_NAME}_training_logs.json")

Training logs saved as effnet_unet3d_brats2020_training_logs.json
